# Data Augmentation con CTGAN — BioCatch Vishing Dataset (1M rows)

## Contexto y motivación

El notebook v2 (copula gaussiano manual) mostró un **AUC de distinguibilidad de 1.00**, lo que significa que un clasificador simple separa perfectamente originales de sintéticos. Diagnóstico probable:

1. **Jitter aditivo sobre features sesgadas positivas** (`session_duration_s`, `transaction_amount_cop`) empujando valores fuera del soporte natural o creando artefactos en las colas.
2. **`searchsorted` sobre CDFs empíricas discretas** replicando exactamente los valores del original en integer columns, con una firma detectable en distribuciones de frecuencia.
3. **Dependencias no lineales** que un copula gaussiano no captura (solo modela correlaciones lineales por pares en el espacio latente).

CTGAN aprende las dependencias directamente vía adversarial training con condicionamiento por modo. En papers y benchmarks (Xu et al., 2019; SDMetrics), típicamente reduce el AUC de distinguibilidad al rango 0.55–0.70 en datasets tabulares mixtos.

## Estrategia de esta implementación

- **Un CTGAN por clase** (legit vs vishing) — la clase minoritaria (2,500 vishing mobile) necesita hiperparámetros específicos porque CTGAN es hambriento de datos.
- **Filtrado a mobile** (igual que v2).
- **Reuso del mismo pipeline de post-processing** de v2: constraints, reglas determinísticas (`is_atypical_hour`), condicional empírica para indicadores BioCatch, recálculo de derivadas.
- **Mismo pipeline de validación** para comparación directa con v2.

## Hiperparámetros de CTGAN

| Clase   | epochs | batch_size | pac | generator/discriminator dims | embedding_dim |
|---------|--------|------------|-----|------------------------------|---------------|
| Legit   | 300    | 500        | 10  | (256, 256)                   | 128           |
| Vishing | 800    | 250        | 5   | (128, 128)                   | 64            |

La configuración de vishing es más conservadora: red más pequeña (menos overfitting con 2,500 muestras), pac menor (packing de menos filas por evaluación del crítico), batch menor y más epochs para convergencia estable.


## 1. Setup e instalación de SDV

In [6]:
import boto3
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
default_bucket = sagemaker_session.default_bucket()

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as scipy_stats

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

import torch

plt.rcParams.update({
    'figure.figsize': (14, 6), 'font.size': 11, 'axes.titlesize': 14,
    'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False,
})
COLORS = {'legit': '#2ecc71', 'vishing': '#e74c3c', 'neutral': '#3498db'}
RNG = np.random.default_rng(42)

# CTGAN se beneficia mucho de GPU
CUDA_AVAILABLE = torch.cuda.is_available()
print(f'PyTorch CUDA available: {CUDA_AVAILABLE}')
if CUDA_AVAILABLE:
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('  ⚠️ Sin GPU. CTGAN va a tardar significativamente más.')
    print('  Recomendado: instance ml.g4dn.xlarge o ml.g5.xlarge en SageMaker.')

print('\n✅ Libraries loaded')

PyTorch CUDA available: True
  GPU: Tesla T4

✅ Libraries loaded


In [8]:
s3_input_path = 's3://poc-fraude-vishing/biocatch_sinthetic_data.csv'
df_original = pd.read_csv(s3_input_path, parse_dates=['session_timestamp'])

df_mobile = df_original[df_original['device_type'] == 'mobile'].copy().reset_index(drop=True)

print(f'Mobile sessions: {len(df_mobile):,}')
print(f'  Legit:   {(df_mobile["is_vishing"]==0).sum():,}')
print(f'  Vishing: {(df_mobile["is_vishing"]==1).sum():,} ({df_mobile["is_vishing"].mean()*100:.2f}%)')

Mobile sessions: 42,579
  Legit:   40,452
  Vishing: 2,127 (5.00%)


## 2. Clasificación de features

Mismo esquema que v2: las features derivadas quedan fuera del entrenamiento de CTGAN (se recalculan al final), `is_atypical_hour` se deriva por regla, y los indicadores BioCatch se generan por condicional empírica sobre `biocatch_risk_score`.

**Nota importante**: CTGAN necesita saber qué columnas son categoricas vs. numéricas vía `SingleTableMetadata`. Las binarias (`phone_call_active`, etc.) las declaramos como `categorical` para que use el condicional discreto (mucho más estable que tratarlas como numéricas).

In [9]:
# Continuas puras
continuous_cols = [
    'avg_keyhold_ms', 'avg_interkey_latency_ms', 'typing_speed_cps',
    'keystroke_variability', 'segmented_typing_ratio',
    'avg_touch_pressure', 'avg_touch_size_px', 'swipe_speed_px_s',
    'swipe_directional_variance', 'scroll_speed_avg',
    'device_tilt_angle_mean', 'device_tilt_variability',
    'gyro_rotation_rate_mean', 'accelerometer_jerk_mean',
    'avg_hesitation_duration_s', 'max_hesitation_duration_s',
    'total_dead_time_s', 'dead_time_ratio',
    'screen_transition_time_avg_s', 'data_familiarity_score',
    'session_duration_s', 'call_overlap_duration_s',
    'time_to_transaction_s',
]

# Derivadas: fuera del CTGAN, se recalculan
derived_cols = ['errors_per_minute', 'interactions_per_s', 'hesitation_composite']

# Integer
integer_cols = [
    'phone_motion_events', 'hesitation_count', 'dead_time_periods',
    'screens_visited', 'unique_screens_visited', 'unusual_screen_visits',
    'navigation_back_count', 'input_error_count', 'input_correction_count',
    'amount_field_corrections', 'beneficiary_field_corrections',
    'copy_paste_events', 'doodling_events', 'hour_of_day',
    'transaction_amount_cop',
    'biocatch_risk_score', 'biocatch_genuine_score',
]

# Binarias que sí entrena CTGAN (como categóricas)
binary_cols_ctgan = [
    'phone_call_active',
    'remote_access_tool_detected', 'suspicious_app_detected',
    'transaction_attempted', 'is_new_beneficiary',
]

# Fuera del CTGAN: se derivan post
rule_binary_cols = ['is_atypical_hour']
conditional_binary_cols = ['biocatch_ato_indicator',
                            'biocatch_social_eng_indicator',
                            'biocatch_bot_indicator']

# Columnas que va a ver CTGAN
ctgan_cols = continuous_cols + integer_cols + binary_cols_ctgan

print(f'Continuas:            {len(continuous_cols)}')
print(f'Integer:              {len(integer_cols)}')
print(f'Binarias (CTGAN):     {len(binary_cols_ctgan)}')
print(f'Total a CTGAN:        {len(ctgan_cols)}')
print(f'Derivadas (post):     {len(derived_cols)}')
print(f'Reglas (post):        {len(rule_binary_cols)}')
print(f'Condicional (post):   {len(conditional_binary_cols)}')

Continuas:            23
Integer:              17
Binarias (CTGAN):     5
Total a CTGAN:        45
Derivadas (post):     3
Reglas (post):        1
Condicional (post):   3


## 3. Helpers de reglas, condicionales y post-processing

Reutilizamos los mismos que en v2 — la lógica de dominio no cambia por el hecho de que el generador subyacente sea distinto.

In [10]:
ATYPICAL_HOURS = {22, 23, 0, 1, 2, 3, 4, 5}
RATIO_COLS_01 = ['segmented_typing_ratio', 'avg_touch_pressure', 'dead_time_ratio',
                 'data_familiarity_score', 'keystroke_variability', 'swipe_directional_variance']

def compute_atypical_hour(hour_series):
    return hour_series.isin(ATYPICAL_HOURS).astype(int)

def build_conditional_indicator_table(df_class, risk_col='biocatch_risk_score',
                                       indicator_cols=('biocatch_ato_indicator',
                                                       'biocatch_social_eng_indicator',
                                                       'biocatch_bot_indicator'),
                                       n_bins=10):
    edges = np.quantile(df_class[risk_col], np.linspace(0, 1, n_bins + 1))
    edges[0] -= 1e-6; edges[-1] += 1e-6
    df_class = df_class.copy()
    df_class['_bin'] = pd.cut(df_class[risk_col], bins=edges, labels=False, include_lowest=True)
    table = {}
    for col in indicator_cols:
        probs = (df_class.groupby('_bin')[col].mean()
                 .reindex(range(n_bins), fill_value=df_class[col].mean()).values)
        table[col] = probs
    return edges, table


def enforce_domain_constraints(df, integer_cols, binary_cols, continuous_cols, mins_reference):
    for col in RATIO_COLS_01:
        if col in df.columns:
            df[col] = df[col].clip(0, 1)
    for col in continuous_cols:
        if col in df.columns and mins_reference.get(col, 0) >= 0:
            df[col] = df[col].clip(lower=0)
    for col in integer_cols:
        if col in df.columns:
            df[col] = np.rint(df[col]).clip(lower=0).astype(int)
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].astype(int).clip(0, 1)
    for col in ['biocatch_risk_score', 'biocatch_genuine_score']:
        if col in df.columns:
            df[col] = df[col].clip(0, 1000).astype(int)
    if 'hour_of_day' in df.columns:
        df['hour_of_day'] = df['hour_of_day'].clip(0, 23).astype(int)
    if 'device_tilt_angle_mean' in df.columns:
        df['device_tilt_angle_mean'] = df['device_tilt_angle_mean'].clip(0, 90)
    return df


def enforce_logical_consistencies(df):
    df['is_atypical_hour'] = compute_atypical_hour(df['hour_of_day']).astype(int)
    df['unique_screens_visited'] = np.minimum(df['unique_screens_visited'], df['screens_visited'])
    df.loc[df['phone_call_active'] == 0, 'call_overlap_duration_s'] = 0.0
    mask_no_tx = df['transaction_attempted'] == 0
    df.loc[mask_no_tx, 'time_to_transaction_s'] = 0.0
    df.loc[mask_no_tx, 'transaction_amount_cop'] = 0
    df.loc[mask_no_tx, 'is_new_beneficiary'] = 0
    df.loc[df['hesitation_count'] == 0, 'avg_hesitation_duration_s'] = 0.0
    df.loc[df['hesitation_count'] == 0, 'max_hesitation_duration_s'] = 0.0
    df['max_hesitation_duration_s'] = np.maximum(df['max_hesitation_duration_s'],
                                                 df['avg_hesitation_duration_s'])
    df.loc[df['dead_time_periods'] == 0, 'total_dead_time_s'] = 0.0
    df.loc[df['dead_time_periods'] == 0, 'dead_time_ratio'] = 0.0
    return df


def apply_biocatch_indicators(df, edges, cond_table, seed=None):
    rng = np.random.default_rng(seed) if seed is not None else RNG
    bins = pd.cut(df['biocatch_risk_score'], bins=edges, labels=False, include_lowest=True)
    bins = bins.fillna(0).astype(int).clip(0, len(edges) - 2)
    for col, probs in cond_table.items():
        p_row = probs[bins.values]
        df[col] = (rng.random(len(df)) < p_row).astype(int)
    return df


def recompute_derived_features(df):
    dur = df['session_duration_s'].clip(lower=1.0)
    df['errors_per_minute'] = df['input_error_count'] / (dur / 60.0)
    interactions = (df['screens_visited'] + df['input_error_count']
                    + df['input_correction_count'] + df['copy_paste_events'])
    df['interactions_per_s'] = interactions / dur
    df['hesitation_composite'] = (df['hesitation_count'] * df['avg_hesitation_duration_s']) / dur
    return df


# Precomputar helpers específicos de la data
days_to_claim_pool = df_mobile.loc[df_mobile['is_vishing']==1, 'days_to_claim'].values
claim_cat_probs = df_mobile.loc[df_mobile['is_vishing']==1, 'claim_category'].value_counts(normalize=True)
edges_legit, cond_table_legit = build_conditional_indicator_table(df_mobile[df_mobile['is_vishing']==0])
edges_vish,  cond_table_vish  = build_conditional_indicator_table(df_mobile[df_mobile['is_vishing']==1])

print('✅ Helpers ready')

✅ Helpers ready


## 4. Metadata SDV

CTGAN necesita tipos declarados por columna. Las binarias las forzamos a `categorical` para que use el mecanismo `conditional_column_conditioning` de CTGAN, que es notablemente más estable que tratarlas como continuas.

In [11]:
def build_metadata(df_class):
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df_class)

    # Forzar sdtype por columna
    for col in continuous_cols:
        if col in df_class.columns:
            metadata.update_column(col, sdtype='numerical', computer_representation='Float')
    for col in integer_cols:
        if col in df_class.columns:
            metadata.update_column(col, sdtype='numerical', computer_representation='Int64')
    for col in binary_cols_ctgan:
        if col in df_class.columns:
            # Categorical con 2 valores => CTGAN usa condicional discreta
            metadata.update_column(col, sdtype='categorical')
    return metadata


# Extraer subconjuntos de entrenamiento
df_train_legit = df_mobile.loc[df_mobile['is_vishing']==0, ctgan_cols].copy().reset_index(drop=True)
df_train_vishing = df_mobile.loc[df_mobile['is_vishing']==1, ctgan_cols].copy().reset_index(drop=True)

# Asegurar tipos: binarias como str para que SDV las detecte categóricas
for col in binary_cols_ctgan:
    df_train_legit[col] = df_train_legit[col].astype(str)
    df_train_vishing[col] = df_train_vishing[col].astype(str)

metadata_legit = build_metadata(df_train_legit)
metadata_vishing = build_metadata(df_train_vishing)

print(f'Legit train shape:   {df_train_legit.shape}')
print(f'Vishing train shape: {df_train_vishing.shape}')
print(f'\nMetadata legit — muestra de sdtypes:')
for col in list(metadata_legit.columns.keys())[:5]:
    print(f'  {col}: {metadata_legit.columns[col]}')

Legit train shape:   (40452, 45)
Vishing train shape: (2127, 45)

Metadata legit — muestra de sdtypes:
  avg_keyhold_ms: {'sdtype': 'numerical', 'computer_representation': 'Float'}
  avg_interkey_latency_ms: {'sdtype': 'numerical', 'computer_representation': 'Float'}
  typing_speed_cps: {'sdtype': 'numerical', 'computer_representation': 'Float'}
  keystroke_variability: {'sdtype': 'numerical', 'computer_representation': 'Float'}
  segmented_typing_ratio: {'sdtype': 'numerical', 'computer_representation': 'Float'}


## 5. Entrenamiento CTGAN — clase legit

47,500 muestras es suficiente para hiperparámetros estándar. Con GPU esto debería tomar entre 5 y 20 minutos según instance. Sin GPU, entre 45 min y 2 horas.

In [12]:
synth_legit = CTGANSynthesizer(
    metadata_legit,
    epochs=300,
    batch_size=500,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    embedding_dim=128,
    pac=10,
    generator_lr=2e-4,
    discriminator_lr=2e-4,
    verbose=True,
    cuda=CUDA_AVAILABLE,
)

print('Training CTGAN on legit class...')
t0 = time.time()
synth_legit.fit(df_train_legit)
print(f'\n✅ Legit CTGAN trained in {(time.time()-t0)/60:.1f} min')

Training CTGAN on legit class...


Gen. (-04.81) | Discrim. (+00.03): 100%|██████████| 300/300 [19:41<00:00,  3.94s/it]


✅ Legit CTGAN trained in 22.0 min


## 6. Entrenamiento CTGAN — clase vishing

Aquí está el trabajo delicado. Con solo 2,500 muestras, hay que:

- Bajar el tamaño de la red (`generator_dim`, `discriminator_dim`, `embedding_dim`) para reducir capacidad y evitar mode collapse.
- Bajar `pac` de 10 a 5 — con muestras escasas, packings grandes desperdician información.
- Bajar `batch_size` a 250 — con 2,500 muestras, batch=500 son 5 batches por epoch, muy pocos pasos de gradiente.
- Subir `epochs` a 800 para compensar los pasos por epoch menores.

Este bloque debería tardar 3-10 min con GPU, 20-40 min sin ella.

In [13]:
synth_vishing = CTGANSynthesizer(
    metadata_vishing,
    epochs=800,
    batch_size=250,
    generator_dim=(128, 128),
    discriminator_dim=(128, 128),
    embedding_dim=64,
    pac=5,
    generator_lr=1e-4,        # Un poco más bajo para más estabilidad
    discriminator_lr=1e-4,
    verbose=True,
    cuda=CUDA_AVAILABLE,
)

print('Training CTGAN on vishing class...')
t0 = time.time()
synth_vishing.fit(df_train_vishing)
print(f'\n✅ Vishing CTGAN trained in {(time.time()-t0)/60:.1f} min')

Training CTGAN on vishing class...


Gen. (-01.32) | Discrim. (+00.20): 100%|██████████| 800/800 [04:36<00:00,  2.89it/s]


✅ Vishing CTGAN trained in 4.8 min


## 7. Generación del dataset 1M

Configuración objetivo: 1M sesiones, ~1.5% vishing.

In [14]:
N_TOTAL = 1_000_000
VISHING_PCT = 0.015

n_vishing_target = int(N_TOTAL * VISHING_PCT)
n_legit_target = N_TOTAL - n_vishing_target

df_orig_legit = df_mobile[df_mobile['is_vishing'] == 0].reset_index(drop=True)
df_orig_vishing = df_mobile[df_mobile['is_vishing'] == 1].reset_index(drop=True)

n_legit_to_generate = max(0, n_legit_target - len(df_orig_legit))
n_vishing_to_generate = max(0, n_vishing_target - len(df_orig_vishing))

print(f'To generate — legit:   {n_legit_to_generate:,}')
print(f'To generate — vishing: {n_vishing_to_generate:,}')

To generate — legit:   944,548
To generate — vishing: 12,873


In [15]:
print('Sampling from legit CTGAN...')
t0 = time.time()
df_synth_legit = synth_legit.sample(num_rows=n_legit_to_generate)
print(f'  ✅ {len(df_synth_legit):,} rows in {(time.time()-t0)/60:.1f} min')

print('\nSampling from vishing CTGAN...')
t0 = time.time()
if n_vishing_to_generate > 0:
    df_synth_vishing = synth_vishing.sample(num_rows=n_vishing_to_generate)
else:
    df_synth_vishing = pd.DataFrame(columns=ctgan_cols)
print(f'  ✅ {len(df_synth_vishing):,} rows in {(time.time()-t0)/60:.1f} min')

# Volver binarias a int (SDV las devolvió como categorical/string)
for col in binary_cols_ctgan:
    df_synth_legit[col] = df_synth_legit[col].astype(int)
    if len(df_synth_vishing) > 0:
        df_synth_vishing[col] = df_synth_vishing[col].astype(int)

Sampling from legit CTGAN...
  ✅ 944,548 rows in 0.7 min

Sampling from vishing CTGAN...
  ✅ 12,873 rows in 0.0 min


## 8. Post-processing: constraints, reglas, derivadas, metadata

El mismo pipeline de v2 — CTGAN produce marginales razonables pero no respeta consistencias lógicas ni constraints de dominio.

In [16]:
def assemble_class(df_orig_class, df_synth_class, label, edges, cond_table):
    mins_ref = df_orig_class[continuous_cols].min().to_dict()

    orig_block = df_orig_class[ctgan_cols].copy()
    # Asegurar binarias a int en los originales
    for col in binary_cols_ctgan:
        orig_block[col] = orig_block[col].astype(int)
    orig_block['is_synthetic'] = 0

    if len(df_synth_class) > 0:
        synth_block = df_synth_class.copy()
        synth_block['is_synthetic'] = 1
        combined = pd.concat([orig_block, synth_block], ignore_index=True)
    else:
        combined = orig_block

    combined = enforce_domain_constraints(combined, integer_cols, binary_cols_ctgan,
                                           continuous_cols, mins_ref)
    combined = enforce_logical_consistencies(combined)
    combined = apply_biocatch_indicators(combined, edges, cond_table, seed=999)
    combined = recompute_derived_features(combined)
    combined['is_vishing'] = label
    return combined


print('Assembling LEGIT block...')
t0 = time.time()
block_legit = assemble_class(df_orig_legit, df_synth_legit, label=0,
                             edges=edges_legit, cond_table=cond_table_legit)
print(f'  ✅ {len(block_legit):,} rows in {time.time()-t0:.1f}s')

print('Assembling VISHING block...')
t0 = time.time()
block_vish = assemble_class(df_orig_vishing, df_synth_vishing, label=1,
                            edges=edges_vish, cond_table=cond_table_vish)
print(f'  ✅ {len(block_vish):,} rows in {time.time()-t0:.1f}s')

df_augmented = pd.concat([block_legit, block_vish], ignore_index=True)
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nTotal augmented: {len(df_augmented):,}')
print(f'Vishing rate: {df_augmented["is_vishing"].mean()*100:.3f}%')

Assembling LEGIT block...
  ✅ 985,000 rows in 0.8s
Assembling VISHING block...
  ✅ 15,000 rows in 0.1s

Total augmented: 1,000,000
Vishing rate: 1.500%


In [17]:
from datetime import datetime, timedelta

n = len(df_augmented)

df_augmented.insert(0, 'session_id', [f'SES-{i:07d}' for i in range(1, n + 1)])
df_augmented.insert(1, 'customer_id',
                    [f'CUS-{RNG.integers(10000, 100000)}' for _ in range(n)])

base_date = datetime(2024, 6, 1)
days_offsets = RNG.integers(0, 365, size=n)
mins = RNG.integers(0, 60, size=n)
secs = RNG.integers(0, 60, size=n)
hours = df_augmented['hour_of_day'].values
timestamps = [base_date + timedelta(days=int(days_offsets[i]),
                                     hours=int(hours[i]),
                                     minutes=int(mins[i]),
                                     seconds=int(secs[i])) for i in range(n)]
df_augmented.insert(2, 'session_timestamp', timestamps)

df_augmented.insert(3, 'device_type', 'mobile')

os_probs = df_mobile['os_type'].value_counts(normalize=True)
df_augmented.insert(4, 'os_type',
                    RNG.choice(os_probs.index.values, size=n, p=os_probs.values))

app_probs = df_mobile['app_version'].value_counts(normalize=True)
df_augmented.insert(5, 'app_version',
                    RNG.choice(app_probs.index.values, size=n, p=app_probs.values))

mask_v = df_augmented['is_vishing'] == 1
n_v = int(mask_v.sum())
df_augmented['days_to_claim'] = -1
df_augmented.loc[mask_v, 'days_to_claim'] = RNG.choice(days_to_claim_pool, size=n_v)

df_augmented['claim_category'] = 'none'
df_augmented.loc[mask_v, 'claim_category'] = RNG.choice(
    claim_cat_probs.index.values, size=n_v, p=claim_cat_probs.values)

print(f'✅ Metadata attached — final shape: {df_augmented.shape}')
print(f'Vishing rate final: {df_augmented["is_vishing"].mean()*100:.3f}%')
print(f'Synthetic rate:     {df_augmented["is_synthetic"].mean()*100:.2f}%')

✅ Metadata attached — final shape: (1000000, 62)
Vishing rate final: 1.500%
Synthetic rate:     95.74%


## 9. Guardado en S3 y persistencia de los CTGANs

In [18]:
output_path = 's3://poc-fraude-vishing/data/augmented/dataset_1M_vishing_ctgan.parquet'
print(f'Saving dataset to {output_path} ...')
t0 = time.time()
df_augmented.to_parquet(output_path, engine='pyarrow', index=False)
print(f'✅ Saved in {time.time()-t0:.1f}s')

# Guardar los synthesizers para re-uso (evita re-entrenar)
os.makedirs('ctgan_models', exist_ok=True)
synth_legit.save('ctgan_models/ctgan_legit.pkl')
synth_vishing.save('ctgan_models/ctgan_vishing.pkl')

# Subir a S3
s3 = boto3.client('s3')
bucket = 'poc-fraude-vishing'
s3.upload_file('ctgan_models/ctgan_legit.pkl', bucket, 'models/ctgan_legit.pkl')
s3.upload_file('ctgan_models/ctgan_vishing.pkl', bucket, 'models/ctgan_vishing.pkl')
print('✅ CTGAN synthesizers saved to S3')

Saving dataset to s3://poc-fraude-vishing/data/augmented/dataset_1M_vishing_ctgan.parquet ...
✅ Saved in 2.4s
✅ CTGAN synthesizers saved to S3
